# LeetCode 3553 — Minimum Weighted Subgraph With the Required Paths II bằng Python

- Link: https://leetcode.com/problems/minimum-weighted-subgraph-with-the-required-paths-ii/
- Độ khó: **Hard**
- Chủ đề: **Cây Steiner của 3 đỉnh trên cây**, **khoảng cách bằng LCA** (binary lifting / Euler tour + sparse table), công thức `(d(a,b) + d(b,c) + d(c,a)) / 2`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumWeight(self, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu "cây con nhỏ nhất để `src1` và `src2` cùng đi tới `dest`" chính là **hợp hai đường đi** `src1 → dest` và `src2 → dest` — cây Steiner nhỏ nhất nối 3 đỉnh.
2. Chứng minh công thức đối xứng đẹp: `trọng số = (d(a,b) + d(b,c) + d(c,a)) / 2`.
3. Tính khoảng cách có trọng số `d(x, y) = D[x] + D[y] − 2·D[lca(x, y)]`.
4. Cài LCA bằng **binary lifting** (`O(log n)`) và **Euler tour + sparse table** (`O(1)` mỗi truy vấn).
5. Kiểm chứng bằng cách đánh dấu cạnh của hai đường đi trên cây nhỏ.

## 2. Đề bài & ví dụ

Cây vô hướng có trọng số `n` đỉnh (`0..n-1`), `edges[i] = [u, v, w]`. Mỗi truy vấn `[src1, src2, dest]` (ba đỉnh đôi một khác nhau): tìm **tổng trọng số nhỏ nhất** của một cây con (tập đỉnh + cạnh liên thông của cây gốc) sao cho từ `src1` và từ `src2` đều đi được tới `dest` chỉ bằng cạnh của cây con đó.

**Ví dụ 1:** `edges=[[0,1,2],[1,2,3],[1,3,5],[1,4,4],[2,5,6]]`, `queries=[[2,3,4],[0,2,5]]` → `[12,11]`.

**Ví dụ 2:** `edges=[[1,0,8],[0,2,7]]`, `queries=[[0,1,2]]` → `[15]`.

**Ràng buộc:** `3 ≤ n ≤ 10^5`, `1 ≤ w ≤ 10^4`, `1 ≤ len(queries) ≤ 10^5`.

In [1]:
from typing import List
from collections import deque
examples = [
    ([[0, 1, 2], [1, 2, 3], [1, 3, 5], [1, 4, 4], [2, 5, 6]], [[2, 3, 4], [0, 2, 5]], [12, 11]),
    ([[1, 0, 8], [0, 2, 7]], [[0, 1, 2]], [15]),
]
for e, q, exp in examples:
    print(e, q, '-> kỳ vọng', exp)

[[0, 1, 2], [1, 2, 3], [1, 3, 5], [1, 4, 4], [2, 5, 6]] [[2, 3, 4], [0, 2, 5]] -> kỳ vọng [12, 11]
[[1, 0, 8], [0, 2, 7]] [[0, 1, 2]] -> kỳ vọng [15]


## 3. Kiến thức nền

### 3.1. Đường đi duy nhất trên cây

Giữa hai đỉnh của cây có **đúng một** đường đi đơn. Muốn `src1` tới được `dest` trong cây con thì cây con phải chứa **toàn bộ** đường `src1 → dest`; tương tự cho `src2`.

### 3.2. Khoảng cách từ gốc và LCA

BFS từ gốc 0: `D[x]` (tổng trọng số tới gốc), `dep[x]` (số cạnh), `par[x]`. Khi đó `d(x, y) = D[x] + D[y] − 2·D[lca(x, y)]`.

In [2]:
def bfs(n, edges):
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
    par, dep, D = [0] * n, [0] * n, [0] * n
    seen = [False] * n
    seen[0] = True
    order = [0]
    for u in order:
        for v, w in adj[u]:
            if not seen[v]:
                seen[v] = True
                par[v], dep[v], D[v] = u, dep[u] + 1, D[u] + w
                order.append(v)
    return par, dep, D, order, adj

E1 = examples[0][0]
par, dep, D, order, adj = bfs(6, E1)
print('par =', par, '| dep =', dep, '| D =', D)

par = [0, 0, 1, 1, 1, 2] | dep = [0, 1, 2, 2, 2, 3] | D = [0, 2, 5, 7, 6, 11]


### 3.3. Binary lifting cho LCA

In [3]:
def build_up(par):
    n = len(par)
    LOG = max(1, (n - 1).bit_length())
    up = [par[:]]
    for _ in range(1, LOG):
        pr = up[-1]
        up.append([pr[pr[x]] for x in range(n)])
    return up

def lca(up, dep, a, b):
    if dep[a] < dep[b]:
        a, b = b, a
    diff, j = dep[a] - dep[b], 0
    while diff:
        if diff & 1:
            a = up[j][a]
        diff >>= 1
        j += 1
    if a == b:
        return a
    for j in range(len(up) - 1, -1, -1):
        if up[j][a] != up[j][b]:
            a, b = up[j][a], up[j][b]
    return up[0][a]

up = build_up(par)

def dist(x, y):
    return D[x] + D[y] - 2 * D[lca(up, dep, x, y)]

print('d(2,3) =', dist(2, 3), '| d(3,4) =', dist(3, 4), '| d(2,4) =', dist(2, 4))

d(2,3) = 8 | d(3,4) = 9 | d(2,4) = 7


### 3.4. Hợp hai đường đi trên cây

Hai đường `a → c` và `b → c` cùng kết thúc tại `c`. Chúng **trùng nhau** trên một đoạn cuối (từ một điểm "giao" `m` tới `c`) và rời nhau trước đó. Điểm `m` được gọi là **trung vị** của ba đỉnh — nó nằm trên cả ba đường `a–b`, `b–c`, `c–a`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `m` là đỉnh trung vị. Ba đường `m → a`, `m → b`, `m → c` rời nhau (trừ `m`). Đặt độ dài của chúng là `x, y, z`. Cây con tối ưu là hợp ba "nhánh" này, trọng số `x + y + z`. Và:

- `d(a, b) = x + y`, `d(b, c) = y + z`, `d(c, a) = z + x`.
- Cộng lại: `d(a,b) + d(b,c) + d(c,a) = 2(x + y + z)`.

⇒ **`trọng số = (d(a,b) + d(b,c) + d(c,a)) / 2`**.

Tính tay ví dụ 1, truy vấn `[2, 3, 4]`: `D = [0, 2, 5, 7, 6, 11]`.

- `d(2,3) = 3 + 5 = 8`, `d(3,4) = 5 + 4 = 9`, `d(2,4) = 3 + 4 = 7`.
- `(8 + 9 + 7) / 2 = 12` ✓.

In [4]:
for a, b, c in [[2, 3, 4], [0, 2, 5]]:
    s = dist(a, b) + dist(b, c) + dist(c, a)
    print(f"[{a},{b},{c}]: d(a,b)={dist(a,b)}, d(b,c)={dist(b,c)}, d(c,a)={dist(c,a)} -> {s} / 2 = {s // 2}")

[2,3,4]: d(a,b)=8, d(b,c)=9, d(c,a)=7 -> 24 / 2 = 12
[0,2,5]: d(a,b)=5, d(b,c)=6, d(c,a)=11 -> 22 / 2 = 11


## 5. Cách 1 — Brute Force: đánh dấu cạnh trên hai đường đi

Với mỗi truy vấn: BFS từ `dest` để có cha (hướng về `dest`), leo từ `src1` và `src2` về `dest`, đánh dấu các cạnh đã đi qua (dùng `set` để không tính trùng), cộng trọng số.

In [5]:
def min_weight_brute(edges, queries):
    n = len(edges) + 1
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
    res = []
    for s1, s2, d in queries:
        prev = {d: (None, 0)}
        dq = deque([d])
        while dq:
            x = dq.popleft()
            for y, w in adj[x]:
                if y not in prev:
                    prev[y] = (x, w)
                    dq.append(y)
        used = set()
        total = 0
        for s in (s1, s2):
            x = s
            while x != d:
                p, w = prev[x]
                key = (min(x, p), max(x, p))
                if key not in used:
                    used.add(key)
                    total += w
                x = p
        res.append(total)
    return res

for e, q, exp in examples:
    print(min_weight_brute(e, q), '| kỳ vọng', exp)

[12, 11] | kỳ vọng [12, 11]
[15] | kỳ vọng [15]


**Độ phức tạp:** `O(n)` mỗi truy vấn ⇒ `O(n · q)` = `10^10` ở giới hạn.

**Nút thắt:** BFS cả cây cho mỗi truy vấn và đi từng cạnh. Công thức ở mục 4 chỉ cần **ba khoảng cách**, mỗi khoảng cách là một LCA.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề Steiner 3 đỉnh trên cây:** cây con nhỏ nhất chứa `a, b, c` có trọng số `(d(a,b) + d(b,c) + d(c,a)) / 2`. Mà cây con thỏa đề phải chứa hai đường `a → c`, `b → c` — hợp của chúng chính là cây Steiner đó (mọi cây con liên thông chứa `a, b, c` đều chứa hợp này; và hợp này đã liên thông).

**Thuật toán:**

1. BFS từ gốc: `par, dep, D`.
2. Binary lifting.
3. Mỗi truy vấn: 3 lần LCA ⇒ 3 khoảng cách ⇒ chia đôi tổng.

Quy tắc ngắn:

- `d(x, y) = D[x] + D[y] − 2·D[lca(x, y)]`.
- Tổng ba khoảng cách luôn **chẵn** (bằng `2(x + y + z)`) ⇒ chia nguyên an toàn.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Leo cha theo độ sâu (không BFS lại)

Tiền xử lý `par, dep, D` một lần; với mỗi truy vấn, tìm LCA bằng cách leo cha từng bước. Dùng công thức ba khoảng cách. `O(h)` mỗi truy vấn.

In [6]:
def min_weight_climb(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)

    def lca_slow(a, b):
        while dep[a] > dep[b]:
            a = par[a]
        while dep[b] > dep[a]:
            b = par[b]
        while a != b:
            a, b = par[a], par[b]
        return a

    def dd(x, y):
        return D[x] + D[y] - 2 * D[lca_slow(x, y)]

    return [(dd(a, b) + dd(b, c) + dd(c, a)) // 2 for a, b, c in queries]

for e, q, exp in examples:
    print(min_weight_climb(e, q), '| kỳ vọng', exp)

[12, 11] | kỳ vọng [12, 11]
[15] | kỳ vọng [15]


### 7.2. Cách 3 — Binary lifting + công thức (lời giải chính)

In [7]:
def min_weight_lifting(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)

    def dd(x, y):
        return D[x] + D[y] - 2 * D[lca(up, dep, x, y)]

    return [(dd(a, b) + dd(b, c) + dd(c, a)) // 2 for a, b, c in queries]

for e, q, exp in examples:
    print(min_weight_lifting(e, q), '| kỳ vọng', exp)

[12, 11] | kỳ vọng [12, 11]
[15] | kỳ vọng [15]


**Độ phức tạp:** `O(n log n)` tiền xử lý + `O(log n)` mỗi truy vấn (3 lần LCA).

### 7.3. Cách 4 — Euler tour + sparse table (LCA `O(1)`)

Duyệt Euler (ghi đỉnh mỗi lần đi vào/quay lại), LCA của `u, v` là đỉnh có độ sâu nhỏ nhất trong đoạn `[first[u], first[v]]` ⇒ RMQ bằng sparse table `O(1)`.

In [8]:
def euler_lca_builder(n, edges):
    par, dep, D, order, _ = bfs(n, edges)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    euler, first = [], [0] * n
    stack = [(0, 0)]
    while stack:
        u, i = stack.pop()
        if i == 0:
            first[u] = len(euler)
        euler.append(u)
        if i < len(children[u]):
            stack.append((u, i + 1))
            stack.append((children[u][i], 0))
    m = len(euler)
    # mã hóa (dep, đỉnh) thành một số để so sánh nhanh: dep * n + đỉnh
    base = [dep[x] * n + x for x in euler]
    sp = [base]
    k = 1
    while (1 << k) <= m:
        prev, h = sp[-1], 1 << (k - 1)
        sp.append([min(prev[i], prev[i + h]) for i in range(m - (1 << k) + 1)])
        k += 1

    def lca_q(u, v):
        l, r = first[u], first[v]
        if l > r:
            l, r = r, l
        j = (r - l + 1).bit_length() - 1
        return min(sp[j][l], sp[j][r - (1 << j) + 1]) % n

    return lca_q, D

def min_weight_euler(edges, queries):
    n = len(edges) + 1
    lca_q, D = euler_lca_builder(n, edges)

    def dd(x, y):
        return D[x] + D[y] - 2 * D[lca_q(x, y)]

    return [(dd(a, b) + dd(b, c) + dd(c, a)) // 2 for a, b, c in queries]

for e, q, exp in examples:
    print(min_weight_euler(e, q), '| kỳ vọng', exp)

[12, 11] | kỳ vọng [12, 11]
[15] | kỳ vọng [15]


**Độ phức tạp:** Euler tour dài `2n − 1`; sparse table `O(n log n)` bộ nhớ; mỗi truy vấn `O(1)` (3 lần RMQ).

### 7.4. Cách 5 — Công thức qua "trung vị" (dùng ba LCA)

Một góc nhìn khác: trong ba LCA `lca(a,b), lca(b,c), lca(c,a)`, có hai cái trùng nhau và cái **sâu nhất** chính là trung vị `m`. Khi đó đáp án = `d(m, a) + d(m, b) + d(m, c)`. Kết quả giống hệt công thức đối xứng.

In [9]:
def min_weight_median(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)
    L = lambda x, y: lca(up, dep, x, y)
    dd = lambda x, y: D[x] + D[y] - 2 * D[L(x, y)]
    res = []
    for a, b, c in queries:
        m = max((L(a, b), L(b, c), L(c, a)), key=dep.__getitem__)
        res.append(dd(m, a) + dd(m, b) + dd(m, c))
    return res

for e, q, exp in examples:
    print(min_weight_median(e, q), '| kỳ vọng', exp)

[12, 11] | kỳ vọng [12, 11]
[15] | kỳ vọng [15]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: in LCA từng cặp, ba khoảng cách, trung vị và đáp án; đối chiếu với tập cạnh thật sự được chọn (cách 1).

In [10]:
def trace(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, adj = bfs(n, edges)
    up = build_up(par)
    print('D =', D, '| dep =', dep)
    for a, b, c in queries:
        pairs = [(a, b), (b, c), (c, a)]
        info = []
        tot = 0
        for x, y in pairs:
            l = lca(up, dep, x, y)
            d = D[x] + D[y] - 2 * D[l]
            tot += d
            info.append(f"lca({x},{y})={l}, d={d}")
        m = max((lca(up, dep, x, y) for x, y in pairs), key=dep.__getitem__)
        print(f"[{a},{b},{c}]: " + '; '.join(info) + f" -> tổng {tot}, đáp án {tot // 2}, trung vị m = {m}")
        # cạnh thật sự (dùng cha theo gốc 0): hợp hai đường a->c, b->c
        def path_edges(x, y):
            l = lca(up, dep, x, y)
            es = set()
            for z in (x, y):
                while z != l:
                    es.add((min(z, par[z]), max(z, par[z])))
                    z = par[z]
            return es
        es = path_edges(a, c) | path_edges(b, c)
        print('    cạnh được chọn:', sorted(es))

trace(*examples[0][:2])

D = [0, 2, 5, 7, 6, 11] | dep = [0, 1, 2, 2, 2, 3]
[2,3,4]: lca(2,3)=1, d=8; lca(3,4)=1, d=9; lca(4,2)=1, d=7 -> tổng 24, đáp án 12, trung vị m = 1
    cạnh được chọn: [(1, 2), (1, 3), (1, 4)]
[0,2,5]: lca(0,2)=0, d=5; lca(2,5)=2, d=6; lca(5,0)=0, d=11 -> tổng 22, đáp án 11, trung vị m = 2
    cạnh được chọn: [(0, 1), (1, 2), (2, 5)]


Ở truy vấn `[2,3,4]`, trung vị là đỉnh 1: ba nhánh `1–2 (3)`, `1–3 (5)`, `1–4 (4)` ⇒ `12`.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def minimumWeight(self, edges: List[List[int]], queries: List[List[int]]) -> List[int]:
        n = len(edges) + 1
        adj = [[] for _ in range(n)]
        for u, v, w in edges:
            adj[u].append((v, w))
            adj[v].append((u, w))
        par, dep, D = [0] * n, [0] * n, [0] * n
        seen = [False] * n
        seen[0] = True
        order = [0]
        for u in order:
            for v, w in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    par[v], dep[v], D[v] = u, dep[u] + 1, D[u] + w
                    order.append(v)
        LOG = max(1, (n - 1).bit_length())
        up = [par]
        for _ in range(1, LOG):
            pr = up[-1]
            up.append([pr[pr[x]] for x in range(n)])

        def lca(a, b):
            if dep[a] < dep[b]:
                a, b = b, a
            diff, j = dep[a] - dep[b], 0
            while diff:
                if diff & 1:
                    a = up[j][a]
                diff >>= 1
                j += 1
            if a == b:
                return a
            for j in range(LOG - 1, -1, -1):
                if up[j][a] != up[j][b]:
                    a, b = up[j][a], up[j][b]
            return par[a]

        res = []
        for a, b, c in queries:
            s = D[a] + D[b] - 2 * D[lca(a, b)]
            s += D[b] + D[c] - 2 * D[lca(b, c)]
            s += D[c] + D[a] - 2 * D[lca(c, a)]
            res.append(s // 2)
        return res

for e, q, exp in examples:
    got = Solution().minimumWeight(e, q)
    print(got, 'OK' if got == exp else 'SAI')

[12, 11] OK
[15] OK


## 10. Vì sao thuật toán đúng

1. **Cây con cần thiết:** mọi cây con hợp lệ phải chứa đường `a → c` và `b → c` (đường duy nhất trên cây). Hợp `P` của hai đường này liên thông (cùng chứa `c`) và là một cây con hợp lệ ⇒ `P` là tối ưu, trọng số tối thiểu = trọng số của `P`.
2. **Cấu trúc của `P`:** tồn tại đỉnh `m` (trung vị) nằm trên cả ba đường `a–b`, `b–c`, `c–a`; `P` là hợp ba nhánh rời nhau `m–a`, `m–b`, `m–c` với độ dài `x, y, z`.
3. **Công thức:** `d(a,b) = x + y`, `d(b,c) = y + z`, `d(c,a) = z + x` ⇒ tổng `= 2(x+y+z)` ⇒ trọng số `P = tổng / 2`.
4. **Khoảng cách qua LCA:** đường `x → y` đi lên tới LCA rồi đi xuống ⇒ `D[x] − D[l] + D[y] − D[l]`.
5. **Trung vị = LCA sâu nhất** trong ba LCA (cách 5): hai trong ba LCA bằng nhau và nằm trên trung vị; cái còn lại chính là trung vị.

## 11. Phân tích Time / Space Complexity

Gọi `n` số đỉnh, `q` số truy vấn.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. BFS + đánh dấu cạnh | — | `O(n)` | `O(n)` |
| 2. Leo cha | `O(n)` | `O(h)` | `O(n)` |
| 3. Binary lifting | `O(n log n)` | `O(log n)` | `O(n log n)` |
| 4. Euler + sparse table | `O(n log n)` | `O(1)` | `O(n log n)` |
| 5. Trung vị | `O(n log n)` | `O(log n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {
    'climb': min_weight_climb,
    'lifting': min_weight_lifting,
    'euler': min_weight_euler,
    'median': min_weight_median,
    'submit': lambda e, q: Solution().minimumWeight(e, q),
}
edge = [
    ([[0, 1, 1], [1, 2, 1]], [[0, 2, 1], [0, 1, 2], [1, 2, 0]], [2, 2, 2]),     # đường thẳng 3 đỉnh
    ([[0, 1, 10000], [0, 2, 10000], [0, 3, 10000]], [[1, 2, 3]], [30000]),     # hình sao
    ([[0, 1, 5], [1, 2, 7], [2, 3, 1]], [[0, 1, 3], [3, 0, 2]], [13, 13]),     # dest nằm giữa
]
for e, q, exp in examples + edge:
    for name, f in impls.items():
        got = f(e, q)
        assert got == exp, (name, e, q, got, exp)
    assert min_weight_brute(e, q) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
random.seed(3553)

def rand_tree(n, wmax=20):
    perm = list(range(n))
    random.shuffle(perm)
    es = []
    for i in range(1, n):
        a, b = perm[random.randrange(i)], perm[i]
        if random.random() < 0.5:
            a, b = b, a
        es.append([a, b, random.randint(1, wmax)])
    return es

cnt = 0
for _ in range(400):
    n = random.randint(3, 14)
    e = rand_tree(n)
    q = [random.sample(range(n), 3) for _ in range(8)]
    exp = min_weight_brute(e, q)
    for name, f in impls.items():
        assert f(e, q) == exp, (name, e, q)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 10^5`)

In [14]:
import time
random.seed(9)
n = 10**5
qs = [random.sample(range(n), 3) for _ in range(n)]
for name, e in [('ngẫu nhiên', rand_tree(n, 10**4)),
                ('đường thẳng', [[i, i + 1, random.randint(1, 10**4)] for i in range(n - 1)])]:
    t = time.time(); a = Solution().minimumWeight(e, qs); t1 = time.time() - t
    t = time.time(); b = min_weight_euler(e, qs); t2 = time.time() - t
    assert a == b
    print(f"{name:>11}: binary lifting {t1:.2f}s | Euler + sparse table {t2:.2f}s")

 ngẫu nhiên: binary lifting 1.34s | Euler + sparse table 0.87s


đường thẳng: binary lifting 1.21s | Euler + sparse table 0.80s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng hai khoảng cách `d(a, c) + d(b, c)` (đếm trùng đoạn chung)

In [15]:
def wrong_sum_two(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)
    dd = lambda x, y: D[x] + D[y] - 2 * D[lca(up, dep, x, y)]
    return [dd(a, c) + dd(b, c) for a, b, c in queries]

e, q, exp = examples[0]
print('sai :', wrong_sum_two(e, q))
print('đúng:', Solution().minimumWeight(e, q))

sai : [16, 17]
đúng: [12, 11]


**Vì sao sai:** hai đường `a → c` và `b → c` dùng chung đoạn `m → c`; cộng thẳng sẽ tính đoạn đó hai lần.

### Lỗi 2 — Dùng `max` hoặc `d(a,b) + d(b,c)` thay vì nửa tổng ba khoảng cách

In [16]:
def wrong_two_of_three(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)
    dd = lambda x, y: D[x] + D[y] - 2 * D[lca(up, dep, x, y)]
    return [min(dd(a, b) + dd(b, c), dd(b, c) + dd(c, a), dd(c, a) + dd(a, b)) for a, b, c in queries]

print('sai :', wrong_two_of_three(e, q))
print('đúng:', Solution().minimumWeight(e, q))

sai : [15, 11]
đúng: [12, 11]


**Vì sao sai:** tổng hai khoảng cách vẫn đếm trùng một nhánh (`= x + 2y + z`); phải lấy nửa tổng **ba** khoảng cách.

### Lỗi 3 — Coi `D[x]` là số cạnh (quên trọng số)

In [17]:
def wrong_unweighted(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)
    dd = lambda x, y: dep[x] + dep[y] - 2 * dep[lca(up, dep, x, y)]     # SAI: đếm cạnh
    return [(dd(a, b) + dd(b, c) + dd(c, a)) // 2 for a, b, c in queries]

print('sai :', wrong_unweighted(e, q))
print('đúng:', Solution().minimumWeight(e, q))

sai : [3, 3]
đúng: [12, 11]


**Vì sao sai:** đề hỏi **tổng trọng số**, không phải số cạnh.

### Lỗi 4 — Coi cạnh `[u, v, w]` là có hướng `u → v` khi dựng cây

In [18]:
def wrong_directed(edges, queries):
    n = len(edges) + 1
    children = [[] for _ in range(n)]
    for u, v, w in edges:
        children[u].append((v, w))                    # SAI
    order, seen = [0], {0}
    for u in order:
        for v, w in children[u]:
            if v not in seen:
                seen.add(v); order.append(v)
    return f"chỉ duyệt được {len(order)}/{n} đỉnh từ gốc 0"

print('sai :', wrong_directed(*examples[1][:2]))
print('đúng:', Solution().minimumWeight(*examples[1][:2]))

sai : chỉ duyệt được 2/3 đỉnh từ gốc 0
đúng: [15]


**Vì sao sai:** ví dụ 2 có cạnh `[1, 0, 8]`; cây vô hướng nên phải thêm cả hai chiều.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cây con nhỏ nhất chứa vài đỉnh cho trước" trên **cây** ⇒ cây Steiner = hợp các đường đi; với 3 đỉnh có công thức **nửa tổng ba khoảng cách**.
- Khoảng cách nhiều lần ⇒ `D[x] + D[y] − 2·D[lca]`.
- Nhiều truy vấn LCA ⇒ binary lifting (dễ cài) hoặc Euler + sparse table (`O(1)`).
- Tổng ba khoảng cách luôn chẵn ⇒ dùng `// 2` không sợ sai số.

## 15. Biến thể của bài

1. Bản I (LC 2203, đồ thị có hướng tổng quát) ⇒ ba lần Dijkstra (từ `src1`, `src2` và ngược từ `dest`), thử mọi điểm gặp nhau.
2. Cây Steiner của `k` đỉnh trên cây ⇒ sắp các đỉnh theo thứ tự Euler (`tin`), đáp án = nửa tổng khoảng cách giữa các đỉnh liên tiếp theo vòng.
3. Hỏi **số cạnh** thay vì trọng số ⇒ dùng `dep`.
4. Hỏi đỉnh "gặp nhau" (trung vị) ⇒ LCA sâu nhất trong ba LCA.

In [19]:
# Biến thể 2: cây Steiner của k đỉnh = nửa tổng khoảng cách vòng theo thứ tự Euler
def steiner_k(edges, nodes):
    n = len(edges) + 1
    par, dep, D, order, _ = bfs(n, edges)
    up = build_up(par)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    tin, t, stack = [0] * n, 0, [0]
    while stack:
        u = stack.pop()
        tin[u] = t; t += 1
        stack.extend(reversed(children[u]))
    nodes = sorted(set(nodes), key=tin.__getitem__)
    dd = lambda x, y: D[x] + D[y] - 2 * D[lca(up, dep, x, y)]
    return sum(dd(nodes[i], nodes[(i + 1) % len(nodes)]) for i in range(len(nodes))) // 2

def steiner_brute(edges, nodes):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)
    used, total = set(), 0
    w = {}
    for u, v, ww in edges:
        w[(min(u, v), max(u, v))] = ww
    root = nodes[0]
    for s in nodes[1:]:
        l = lca(up, dep, s, root)
        for z in (s, root):
            while z != l:
                key = (min(z, par[z]), max(z, par[z]))
                if key not in used:
                    used.add(key); total += w[key]
                z = par[z]
    return total

random.seed(15)
for _ in range(200):
    n = random.randint(3, 15)
    e = rand_tree(n)
    nodes = random.sample(range(n), random.randint(2, min(6, n)))
    assert steiner_k(e, nodes) == steiner_brute(e, nodes)
print('Biến thể 2 khớp vét cạn; ví dụ:', steiner_k(E1, [0, 3, 4, 5]))

Biến thể 2 khớp vét cạn; ví dụ: 20


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | BFS + đánh dấu cạnh | `O(n)` | `O(n)` | Oracle |
| 2 | Leo cha + công thức | `O(h)` | `O(n)` | Cây nông |
| 3 | Binary lifting + công thức | `O(log n)` | `O(n log n)` | **Nộp bài** |
| 4 | Euler + sparse table | `O(1)` | `O(n log n)` | Rất nhiều truy vấn |
| 5 | Trung vị (LCA sâu nhất) | `O(log n)` | `O(n log n)` | Khi cần chính đỉnh gặp nhau |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** ví dụ 1, truy vấn `[3, 5, 0]`.

**Bài 2 (tính tay):** đường thẳng `0–1–2–3–4` trọng số `[1, 2, 3, 4]`, truy vấn `[0, 4, 2]`.

**Bài 3 (điền TODO):** viết `meeting_point(a, b, c)` trả về trung vị.

**Bài 4 (tự viết lại):** biến thể 3 — đếm **số cạnh** của cây con tối ưu.

**Bài 5 (mini challenge):** trả lời truy vấn cho **4 đỉnh** `[a, b, c, d]` bằng biến thể 2 và kiểm chứng bằng đánh dấu cạnh.

In [20]:
def meeting_point_todo(up, dep, a, b, c):
    # TODO: LCA sâu nhất trong ba LCA
    raise NotImplementedError

def count_edges_todo(edges, queries):
    # TODO
    raise NotImplementedError

def steiner4_todo(edges, quads):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `D = [0,2,5,7,6,11]`. (2) `d(3,5) = 5+3+6 = 14`, `d(5,0) = 11`, `d(0,3) = 7`. (3) `(14+11+7)/2 = 16`.

**Bài 2.** (1) `d(0,4) = 10`, `d(4,2) = 7`, `d(2,0) = 3`. (2) Tổng 20. (3) Đáp án 10 (chính là đường `0 → 4` vì 2 nằm trên đó).

**Bài 3.** (1) Tính ba LCA. (2) Chọn cái có `dep` lớn nhất. (3) Kiểm tra nó nằm trên cả ba đường (`d(a,m) + d(m,b) = d(a,b)`...).

**Bài 4.** (1) Thay `D` bằng `dep`. (2) Công thức giữ nguyên. (3) So với đánh dấu cạnh.

**Bài 5.** (1) Sắp 4 đỉnh theo `tin`. (2) Nửa tổng khoảng cách vòng. (3) Đánh dấu cạnh của hợp các đường từ đỉnh đầu tới các đỉnh còn lại.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
a1 = Solution().minimumWeight(E1, [[3, 5, 0]])
chain = [[0, 1, 1], [1, 2, 2], [2, 3, 3], [3, 4, 4]]
a2 = Solution().minimumWeight(chain, [[0, 4, 2]])
assert a1 == [16] == min_weight_brute(E1, [[3, 5, 0]])
assert a2 == [10] == min_weight_brute(chain, [[0, 4, 2]])
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: [16] | Bài 2: [10]


In [22]:
# Bài 3, 4
def meeting_point(up, dep, a, b, c):
    return max((lca(up, dep, a, b), lca(up, dep, b, c), lca(up, dep, c, a)), key=dep.__getitem__)

def count_edges(edges, queries):
    n = len(edges) + 1
    par, dep, D, _, _ = bfs(n, edges)
    up = build_up(par)
    dd = lambda x, y: dep[x] + dep[y] - 2 * dep[lca(up, dep, x, y)]
    return [(dd(a, b) + dd(b, c) + dd(c, a)) // 2 for a, b, c in queries]

random.seed(16)
for _ in range(200):
    n = random.randint(3, 14)
    e = rand_tree(n)
    par, dep, D, _, _ = bfs(n, e)
    up = build_up(par)
    dd = lambda x, y: D[x] + D[y] - 2 * D[lca(up, dep, x, y)]
    a, b, c = random.sample(range(n), 3)
    m = meeting_point(up, dep, a, b, c)
    assert dd(a, m) + dd(m, b) == dd(a, b) and dd(b, m) + dd(m, c) == dd(b, c) and dd(c, m) + dd(m, a) == dd(c, a)
    unit = [[u, v, 1] for u, v, _ in e]
    assert count_edges(e, [[a, b, c]]) == min_weight_brute(unit, [[a, b, c]])
print('Bài 3: trung vị nằm trên cả ba đường | Bài 4: đếm cạnh đúng')

Bài 3: trung vị nằm trên cả ba đường | Bài 4: đếm cạnh đúng


In [23]:
# Bài 5: 4 đỉnh
def steiner4(edges, quads):
    return [steiner_k(edges, list(qd)) for qd in quads]

random.seed(17)
for _ in range(150):
    n = random.randint(4, 14)
    e = rand_tree(n)
    qd = random.sample(range(n), 4)
    assert steiner4(e, [qd]) == [steiner_brute(e, qd)]
print('Bài 5: ví dụ', steiner4(E1, [[0, 3, 4, 5], [2, 3, 4, 5]]))

Bài 5: ví dụ [20, 18]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
BFS từ 0: par, dep, D (tổng trọng số);  binary lifting up[j]
d(x, y) = D[x] + D[y] - 2 * D[lca(x, y)]
đáp án(a, b, c) = (d(a,b) + d(b,c) + d(c,a)) // 2
```

**Checklist:**

- [ ] Cạnh vô hướng (thêm hai chiều)?
- [ ] Dùng `D` (trọng số), không phải `dep`?
- [ ] Nửa tổng **ba** khoảng cách (không cộng hai đường)?
- [ ] LCA không đệ quy, chạy được `n = 10^5`?
- [ ] Đối chiếu với đánh dấu cạnh trên cây nhỏ?